# Этап 1. Загрузка данных

Чтение и сохранение исходного файла `all_regions.csv` (~2 ГБ) через DuckDB, явное задание типов колонок и сохранение результата в Parquet.

## Исходные данные

| Колонка | Тип | Описание |
| --- | --- | --- |
| `brand`, `name`, `bodyType`, `color`, `fuelType`, `transmission`, `vehicleConfiguration`, `engineName`, `engineDisplacement`, `location`, `link`, `description` | `VARCHAR` | Текст |
| `year` | `SMALLINT` | Год выпуска |
| `mileage` | `INTEGER` | Пробег, км |
| `power` | `SMALLINT` | Мощность, л.с. |
| `price` | `BIGINT` | Цена, руб. |
| `date` | `TIMESTAMP` | Дата публикации объявления |
| `parse_date` | `TIMESTAMP` | Дата сбора данных |

In [12]:
import duckdb

con = duckdb.connect()
RAW_CSV = "all_regions.csv"
OUTPUT_PARQUET = "data/processed/raw_typed.parquet"

RAW_CSV, OUTPUT_PARQUET


('all_regions.csv', 'data/processed/raw_typed.parquet')

In [ ]:
con.execute(
    """
    CREATE OR REPLACE TABLE cars_raw AS
    SELECT
        brand,
        name,
        bodyType,
        color,
        fuelType,
        CAST(year AS SMALLINT) AS year,
        CAST(mileage AS INTEGER) AS mileage,
        transmission,
        CAST(power AS SMALLINT) AS power,
        CAST(price AS BIGINT) AS price,
        vehicleConfiguration,
        engineName,
        engineDisplacement,
        CAST(date AS TIMESTAMP) AS date,
        location,
        link,
        description,
        CAST(parse_date AS TIMESTAMP) AS parse_date
    FROM read_csv(
        ?,
        header = true,
        delim = ',',
        quote = '"',
        escape = '"',
        encoding = 'utf-8',
        all_varchar = true
    )
    """,
    [str(RAW_CSV)],
)

con.execute(
    f"COPY cars_raw TO '{OUTPUT_PARQUET.as_posix()}' (FORMAT parquet, COMPRESSION zstd)"
)

con.sql("SELECT count(*) AS rows_loaded FROM cars_raw").df()


,rows_loaded
0,1294757


In [3]:
con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(OUTPUT_PARQUET)],
).df()


,column_name,column_type,null,key,default,extra
0,brand,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,bodyType,VARCHAR,YES,None,None,None
3,color,VARCHAR,YES,None,None,None
4,fuelType,VARCHAR,YES,None,None,None
5,year,SMALLINT,YES,None,None,None
6,mileage,INTEGER,YES,None,None,None
7,transmission,VARCHAR,YES,None,None,None
8,power,SMALLINT,YES,None,None,None
9,price,BIGINT,YES,None,None,None


In [4]:
import pandas as pd

size_mb = OUTPUT_PARQUET.stat().st_size / 1024**2
pd.DataFrame(
    {
        "file": [str(OUTPUT_PARQUET)],
        "size_mb": [round(size_mb, 1)],
    }
)

,file,size_mb
0,E:\Dev\car-market-analysis\data\processed\raw_...,224.5


CSV прочитан DuckDB без загрузки в память, числовые и временные колонки приведены к явным типам, результат сохранён в `data/processed/raw_typed.parquet` со сжатием zstd.